# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sohamrathi17/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

**How the queue is built** (everything from ML-04 to ML-09, put together):

- **Score** = model probability of decline × (1 + impressions lost per day within March). The model is the logistic regression from ML-08, trained the way it would be used for real: on the latest month whose outcome is already known (February → March), then applied to March features. ML-09 showed this ordering holds on a month it was not tuned on.
- **Rank** inside each client: the client's editor reads their own list from the top.
- **Action** (one per page, plain rules on March numbers only):

| Action | When | What the editor does |
|---|---|---|
| **refresh** | sliding (late-March impressions under 80% of early-March) and average position ≤ 10 | update facts, examples, title/meta on a page-one page that is losing ground |
| **expand** | sliding and position 11-20 | add missing subtopics and depth to push a near-page-one page back up |
| **monitor** | the slide looks like a one-day spike fading, or position > 20, or the drop is small | do nothing yet, re-check next month |
| **protect** | holding steady (late ≥ 95% of early) and position ≤ 10 | leave it alone, do not "fix" what works |

- **Reason codes** (up to two per page, in words a person can check in Search Console): how much the page slid inside March, CTR below its position tier, a single-day spike, position.
- **Confidence** label: **high** when the client has 300+ pages, the slide is real (momentum < 0.8), not spike-shaped, with 1,000+ March impressions. **low** for clients under 100 pages, spike-shaped slides, or pages that are not sliding. **medium** for the rest. ML-08 showed small clients get no lift and spikes cause most misses, which is where these cut-offs come from.

In [1]:
import os, sys, subprocess, json, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
# numpy 2 + Apple Accelerate prints spurious "overflow in matmul" warnings inside sklearn; results are unaffected
warnings.filterwarnings("ignore", category=RuntimeWarning, module="sklearn")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sohamrathi17/flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
else:
    while not os.path.isdir("work/scripts") and os.getcwd() != "/":
        os.chdir("..")
sys.path.insert(0, "work/scripts")

import numpy as np, pandas as pd
import lane_data as L

# Deployment-shaped setup. On 31 March the newest month with a KNOWN outcome is Feb -> Mar,
# so the model is trained on that, then ranks pages using March features.
# April is used only afterwards, to grade the queue.
train = L.add_drop_feature(L.build_frame("2026-02", "2026-03"))
queue = L.add_drop_feature(L.build_frame("2026-03", "2026-04"))
model = L.make_logistic().fit(train[L.MODEL_FEATURES], train["is_declining"])
queue["p_decline"] = model.predict_proba(queue[L.MODEL_FEATURES])[:, 1]
queue["drop_per_day"] = L.drop_per_day(queue)
queue["score"] = queue["p_decline"] * (1 + queue["drop_per_day"])
print(f"Trained on {len(train):,} pages (Feb -> Mar). Queue: {len(queue):,} pages from {queue['client_hash_id'].nunique()} clients.")

Trained on 43,596 pages (Feb -> Mar). Queue: 50,299 pages from 32 clients.


In [2]:
tier = pd.cut(queue["avg_position"], L.TIER_EDGES, labels=L.TIER_LABELS)
tier_ctr = queue.groupby(tier, observed=True)["ctr"].median()
queue["ctr_vs_tier"] = queue["ctr"] / tier.map(tier_ctr).astype(float)
pages_in_client = queue.groupby("client_hash_id")["content_hash_id"].transform("size")

sliding = queue["momentum"] < 0.8
holding = queue["momentum"] >= 0.95
spiky   = queue["top_day_share"] >= 0.2
pos     = queue["avg_position"]

queue["action"] = np.select(
    [sliding & ~spiky & (pos <= 10), sliding & ~spiky & (pos > 10) & (pos <= 20), holding & (pos <= 10)],
    ["refresh", "expand", "protect"], "monitor")

def reasons(r):
    out = []
    if r["momentum"] < 0.8:
        out.append(f"daily impressions fell {1 - r['imp_late_per_day'] / max(r['imp_early_per_day'], 1):.0%} within March")
    if r["clicks"] > 0 and r["ctr_vs_tier"] < 1:
        out.append("CTR below its position tier")
    if r["top_day_share"] >= 0.2:
        out.append(f"{r['top_day_share']:.0%} of March impressions on one day (spike)")
    if r["avg_position"] <= 10:
        out.append(f"page 1 (avg position {r['avg_position']:.1f})")
    elif r["avg_position"] <= 20:
        out.append(f"striking distance (avg position {r['avg_position']:.1f})")
    return "; ".join(out[:2]) if out else "holding steady"

queue["confidence"] = np.select(
    [(pages_in_client >= 300) & sliding & ~spiky & (queue["imp"] >= 1000),
     (pages_in_client < 100) | spiky | ~sliding],
    ["high", "low"], "medium")

queue = queue.sort_values(["client_hash_id", "score", "content_hash_id"], ascending=[True, False, True])
queue["rank_in_client"] = queue.groupby("client_hash_id").cumcount() + 1
top20 = queue[queue["rank_in_client"] <= 20].copy()
top20["reason_codes"] = top20.apply(reasons, axis=1)
queue["reason_codes"] = ""
queue.loc[top20.index, "reason_codes"] = top20["reason_codes"]

print("Actions in everyone's top 20:", top20["action"].value_counts().to_dict())
print("Confidence in everyone's top 20:", top20["confidence"].value_counts().to_dict())

# One example client's queue (client shown only as a letter, no IDs)
big = pages_in_client.groupby(queue["client_hash_id"]).first().sort_values(ascending=False)
example = top20[top20["client_hash_id"] == big.index[3]].head(10)
example.assign(march_imp=example["imp"].astype(int), position=example["avg_position"].round(1), score=example["score"].round(1))[
    ["rank_in_client", "action", "confidence", "reason_codes", "march_imp", "position", "score"]].set_index("rank_in_client")

Actions in everyone's top 20: {'refresh': 255, 'monitor': 169, 'expand': 49, 'protect': 32}
Confidence in everyone's top 20: {'high': 215, 'low': 186, 'medium': 104}


,action,confidence,reason_codes,march_imp,position,score
rank_in_client,,,,,,
1,monitor,high,daily impressions fell 64% within March,16428,40.4,233.9
2,refresh,high,daily impressions fell 50% within March; page ...,11355,6.9,225.2
3,monitor,low,daily impressions fell 52% within March; 30% o...,12930,2.7,205.3
4,monitor,high,daily impressions fell 60% within March; CTR b...,13806,28.1,160.8
5,monitor,high,daily impressions fell 39% within March,26406,27.0,114.9
6,monitor,high,daily impressions fell 44% within March; CTR b...,14118,28.0,103.7
7,monitor,low,daily impressions fell 77% within March; 58% o...,2574,2.0,98.7
8,refresh,high,daily impressions fell 23% within March; CTR b...,17465,2.9,94.3
9,refresh,high,daily impressions fell 44% within March; page ...,6821,8.5,85.8


**Did the queue work?** Graded afterwards on April (the editor would not have this on 31 March). Precision = share of the pages in each group that really declined in April.

In [3]:
eligible = queue.groupby("client_hash_id")["content_hash_id"].transform("size") >= 20
g = top20[top20["client_hash_id"].isin(queue.loc[eligible, "client_hash_id"])]
overall = L.summarize(queue, queue["score"])
rule = L.summarize(queue, L.baseline_score(queue))
print(f"Queue (top 20 per client, 22 clients): precision {overall['precision@20']:.3f}, loss captured {overall['loss_captured@20']:.3f}"
      f" | ML-07 rule: {rule['precision@20']:.3f} / {rule['loss_captured@20']:.3f} | base rate {overall['base_rate']:.3f}\n")
by_conf = g.groupby("confidence")["is_declining"].agg(pages="size", declined="mean").reindex(["high", "medium", "low"])
by_action = g.groupby("action")["is_declining"].agg(pages="size", declined="mean")
print("Top-20 picks by confidence label:\n" + by_conf.round(3).to_string())
print("\nTop-20 picks by action:\n" + by_action.round(3).to_string())

Queue (top 20 per client, 22 clients): precision 0.720, loss captured 0.235 | ML-07 rule: 0.711 / 0.241 | base rate 0.489

Top-20 picks by confidence label:
            pages  declined
confidence                 
high          215     0.786
medium        104     0.721
low           121     0.603

Top-20 picks by action:
         pages  declined
action                  
expand      45     0.711
monitor    142     0.711
protect      2     0.500
refresh    251     0.729


The confidence labels behave the way they should: **high**-confidence picks declined most often and **low** the least, so the label shows where to spend the first hours. One caveat: I set the confidence cut-offs from what ML-08 taught me about the same March→April month, so this grading is not fully independent. The next month's queue is the real test.

Confidence is about *whether the slide is real*, not about the action. **Monitor** pages decline about as often as refresh pages: they are marked monitor because the right action is unclear (deep positions, or a slide that may be a fading spike), not because they are safe. "Protect" pages barely appear in a top 20, as intended: they are listed so editors know *not* to touch them, not as review work.

## 2. Intended use and limits

**Who uses it:** a content editor or SEO lead, once a month, for one client at a time.

**For what:** deciding which 10-20 pages to open first this month, and which action to consider on each. It replaces "sort by traffic" or "review the oldest pages", which ML-03 and ML-07 showed point at the wrong pages.

**Where it stops being valid:**
- **Clients with fewer than ~100 pages.** No measured lift there (ML-08). Their editors can review everything by hand anyway.
- **Pages without full Search Console coverage** for the month, or with under 100 monthly impressions: they are not in the slice and were never tested.
- **As a probability.** ML-09 showed the probabilities do not carry over between months (the decline rate swung from 21% to 47%). Show the rank and the reason codes, never "X% chance to decline".
- **As proof that a refresh works.** The data is observational. The queue says where a slide is visible and likely to continue, not that an action will fix it.
- **Outside this portfolio and period.** Validated on February-April 2026 for these clients only.

## 3. Human review + the no-go list

**Before acting on any page, a person checks:**
1. **Is the slide real?** Open the page's daily impressions in Search Console. A one-day spike that faded (the most common miss in ML-08) means *monitor*, not refresh.
2. **Is it seasonal or demand-driven?** If the topic's search demand dropped (holiday, news cycle), editing the page will not bring it back.
3. **Did something outside the page change?** A site migration, a tracking gap, or a URL change shows up as a "decline" too.
4. **Was this page edited recently?** A page refreshed last month should get time to settle before another edit.
5. **Does the action make sense for this page?** The action label is a starting suggestion, not an order.

**Never automate:**
- Rewriting, merging, redirecting, or deleting pages based on the score.
- Publishing title or meta changes without a human reading them.
- Showing the score to clients as a forecast or a guarantee.
- Using the queue to judge writers or editors.
- Feeding the queue's own past actions back in as features (it would start learning its own decisions, the decision-derived leakage from ML-09).

## 4. Monitoring / retrain triggers

Checked every month before a new queue goes out. The numbers below are the reference values from this run.

In [4]:
checks = {
    "decline base rate (train month -> queue month)": f"{train['is_declining'].mean():.1%} -> {queue['is_declining'].mean():.1%}",
    "share of pages sliding inside the month (momentum < 0.8)": f"{(train['momentum'] < 0.8).mean():.1%} -> {(queue['momentum'] < 0.8).mean():.1%}",
    "pages in the slice": f"{len(train):,} -> {len(queue):,}",
    "clients in the slice": f"{train['client_hash_id'].nunique()} -> {queue['client_hash_id'].nunique()}",
    "queue precision@20 vs ML-07 rule (graded next month)": f"{overall['precision@20']:.3f} vs {rule['precision@20']:.3f}",
}
for k, v in checks.items():
    print(f"{k:58} {v}")

decline base rate (train month -> queue month)             21.1% -> 47.4%
share of pages sliding inside the month (momentum < 0.8)   15.3% -> 30.3%
pages in the slice                                         43,596 -> 50,299
clients in the slice                                       23 -> 32
queue precision@20 vs ML-07 rule (graded next month)       0.720 vs 0.711


**Retrain or stop when:**
- **Precision@20 falls below the ML-07 rule** for two months in a row, once graded. Then the model is adding nothing and the simpler rule should be used.
- **The decline base rate moves by more than ~15 points month to month** (as it did here, 21% → 47%). Ranking survived that once, but it is a sign that something portfolio-wide (seasonality, a search update, a tracking change) is going on. Check it by hand before trusting the queue.
- **The share of sliding pages or the slice size changes sharply.** That usually means a tracking or coverage problem, not a real change in pages.
- **New clients make up a big part of the queue.** ML-09 had only 8 unseen clients to test on, so a portfolio dominated by new sites needs a fresh check.
- **Routine:** retrain every month on the latest month with a known outcome, and keep the final month of any panel sealed for testing.

## 5. Exports for the paper

The full queue goes to `work/outputs/action_queue.csv` (regenerated on every run, gitignored like all data files). Small aggregate numbers for the paper's charts go to `work/outputs/paper_data.json`, which is committed: these are the receipts for every number in the paper.

In [5]:
os.makedirs("work/outputs", exist_ok=True)
QUEUE_COLS = ["client_hash_id", "content_hash_id", "rank_in_client", "score", "action", "confidence", "reason_codes",
              "imp", "avg_position", "ctr", "momentum", "drop_per_day", "top_day_share"]
queue[QUEUE_COLS].to_csv("work/outputs/action_queue.csv", index=False)

# Aggregates for the paper's charts (no IDs, no page-level rows)
mom_bins = [0, 0.5, 0.8, 0.95, 1.05, 2, np.inf]
mom_labels = ["< 0.5", "0.5-0.8", "0.8-0.95", "0.95-1.05", "1.05-2", "> 2"]
mom = queue.groupby(pd.cut(queue["momentum"], mom_bins, labels=mom_labels), observed=True)["is_declining"].agg(["size", "mean"])
clicked = queue[queue["clicks"] > 0]
ctr_bins = [0, 0.5, 1, 2, np.inf]
ctr_labels = ["< 0.5x tier", "0.5-1x", "1-2x", "2x+"]
ctr = clicked.groupby(pd.cut(clicked["ctr_vs_tier"], ctr_bins, labels=ctr_labels, right=False), observed=True)["is_declining"].agg(["size", "mean"])

model_m = json.load(open("work/outputs/model_metrics.json"))["results"]
valid_m = json.load(open("work/outputs/validation_metrics.json"))
paper = {
    "slice": {"pages": int(len(queue)), "clients": int(queue["client_hash_id"].nunique()),
              "decline_rate": round(float(queue["is_declining"].mean()), 4),
              "train_pages": int(len(train)), "train_decline_rate": round(float(train["is_declining"].mean()), 4)},
    "decline_by_momentum": [{"bucket": str(k), "n": int(r["size"]), "rate": round(float(r["mean"]), 4)} for k, r in mom.iterrows()],
    "decline_by_ctr_vs_tier": [{"bucket": str(k), "n": int(r["size"]), "rate": round(float(r["mean"]), 4)} for k, r in ctr.iterrows()],
    "model_comparison": model_m,
    "validation": valid_m["designs"],
    "new_clients_time_aware": valid_m["new_clients_time_aware"],
    "calibration": valid_m["calibration"],
    "leak_test_auc": valid_m["leak_test_auc"],
    "population_dropped_share": valid_m["population_dropped_share"],
    "queue_eval": {"precision@20": round(overall["precision@20"], 4), "loss_captured@20": round(overall["loss_captured@20"], 4),
                   "rule_precision@20": round(rule["precision@20"], 4), "rule_loss_captured@20": round(rule["loss_captured@20"], 4),
                   "base_rate": round(overall["base_rate"], 4), "clients": overall["clients"]},
    "top20_by_confidence": [{"label": k, "n": int(r["pages"]), "rate": round(float(r["declined"]), 4)} for k, r in by_conf.iterrows()],
    "top20_by_action": [{"label": k, "n": int(r["pages"]), "rate": round(float(r["declined"]), 4)} for k, r in by_action.iterrows()],
    "example_queue": example.assign(position=example["avg_position"].round(1))[
        ["rank_in_client", "action", "confidence", "reason_codes", "position"]].to_dict("records"),
}
with open("work/outputs/paper_data.json", "w") as fh:
    json.dump(paper, fh, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
print(f"Wrote work/outputs/action_queue.csv ({len(queue):,} rows) and work/outputs/paper_data.json")

Wrote work/outputs/action_queue.csv (50,299 rows) and work/outputs/paper_data.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled: markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`, then submit your repo URL on the card. Done.